In [4]:
# Comparing of list of Annotated View records with records with metadata from Excel

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, TypedDict, Any, Set
from collections import Counter
import pandas as pd
import sys

def noises_annotated_stats(data: Optional[Dict[str, Any]]) -> Dict[str, int]:
    """
    Extracts `noises_annotated` from parsed JSON dict `data` and returns:
      - noises_annotated_count: number of valid intervals
      - noises_annotated_len_sum: sum of (endIndex - startIndex) across valid intervals

    A valid interval is a dict with numeric startIndex/endIndex where endIndex > startIndex.
    """
    out = {
        "noises_annotated_count": 0,
        "noises_annotated_len_sum": 0,
    }
    if not isinstance(data, dict):
        return out

    raw = data.get("noises_annotated", [])
    if raw is None:
        return out
    if not isinstance(raw, list):
        return out

    count = 0
    total_len = 0

    for item in raw:
        if not isinstance(item, dict):
            continue

        s = item.get("startIndex")
        e = item.get("endIndex")

        # accept ints/floats/str numbers safely
        try:
            s_i = int(float(s))
            e_i = int(float(e))
        except (TypeError, ValueError):
            continue

        if e_i <= s_i:
            continue

        count += 1
        total_len += (e_i - s_i)

    out["noises_annotated_count"] = count
    out["noises_annotated_len_sum"] = total_len
    return out

# funkcija skaito Excel su įrašų sąrašu ir meta duomenimis 
def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    print("\nSkaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    # filtered = df[df["tag"] != "9999"]
    filtered = df
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    print("Įrašų sąraše:", len(names))
    return names, df

def _read_json_data(json_path: Path) -> Optional[Dict[str, Any]]:
    """
    Reads JSON file and returns parsed dict.
    Returns None if file does not exist, JSON is invalid, or top-level is not a dict.
    """
    if not json_path.exists():
        return None

    try:
        with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError):
        return None

    if not isinstance(data, dict):
        return None

    return data

from pathlib import Path
from typing import Any, Dict, Optional, Set, List
import json
import pandas as pd

# assumes these already exist in your code:
# _read_json_data(json_path: Path) -> Optional[Dict[str, Any]]
# _extract_raw_flags_set(data: Dict[str, Any]) -> Optional[Set[str]]

DEFAULT_KNOWN_FLAGS = [
    "PSEUDO_ANNOTATED",
    "FOR_EXTERNAL_ANNOTATING",
    "EXTERNALLY_ANNOTATED",
    "FULLY_ANNOTATED_PROFESSIONALLY",
]

def flags_str_for_filename(
    fname: str,
    rec_dir: Path,
    known_flags: Optional[List[str]] = None,
) -> str:
    """
    Reads <rec_dir>/<fname>.json, extracts flags, and returns a comma-separated string
    of known_flags that are present. Returns "" if missing/invalid.
    """
    known_flags = known_flags or DEFAULT_KNOWN_FLAGS

    json_path = (rec_dir / fname).with_suffix(".json")

    data = _read_json_data(json_path)
    if data is None:
        return ""  # or "MISSING/INVALID"

    raw_flags_set = _extract_raw_flags_set(data)
    if raw_flags_set is None:
        return ""  # invalid flags structure

    return ",".join(sorted(raw_flags_set))


def _extract_raw_flags_set(data: Dict[str, Any]) -> Optional[Set[str]]:
    """
    Extracts flags from JSON data and returns them as a set of strings.
    Returns None if 'flags' exists but is not a list (invalid structure).
    """
    raw_flags = data.get("flags")

    if raw_flags is None:
        raw_flags_list: List[str] = []
    elif isinstance(raw_flags, list):
        raw_flags_list = [str(x) for x in raw_flags]
    else:
        return None  # invalid structure

    return set(raw_flags_list)


csv_path = Path("AnnotatedViewList_2026_01_22.csv")  # or full path
df = pd.read_csv(csv_path)

# If you want a list of timestamps as strings (already like xxxxxxx.xxx)
csv_names = df["timestamp"].astype(str).tolist()

# Filenames with a suffix you choose
# filenames_npy  = [f"{t}.npy" for t in timestamps]
# filenames_json = [f"{t}.json" for t in timestamps]
# filenames_dat  = [f"{t}.dat" for t in timestamps]

print(csv_names)

# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = Path().resolve().parent / "../SUPL_FUNCTIONS"
sys.path.append(str(PARALLEL_PATH))

from project_util import find_project_root_by_name


PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=PROJECT_DIR)
print("\nPROJECT ROOT DIR:", PROJECT_ROOT)
print("PROJECT DIR:", PROJECT_DIR)

LIST_DIR = PROJECT_ROOT / '1_PREPARE_TRAIN_UNET_DATA'/ 'ecg_zive_npy_for_preparing'
REC_DIR = PROJECT_ROOT / "DATA_ORIG/ecg_zive_npy"
# EXCEL_NAME = "visi_zive_irasai_atrankai_test.xlsx"
EXCEL_NAME = "visi_zive_irasai_atrankai.xlsx"

print("\nLIST_DIR:", LIST_DIR)
print("REC_DIR:", REC_DIR)
print("EXCEL_NAME:", EXCEL_NAME)

file_names, df_meta = read_filenames_from_excel(LIST_DIR / EXCEL_NAME)
# print(file_names)

# filter matches
# matches = df_meta[df_meta["basename"].isin(csv_names)].copy()

# print(f"CSV files: {len(csv_names)}")
print(f"Excel rows: {len(df_meta)}")
# print(f"Matches: {len(matches)}")

def noises_annotated_stats_from_filename(fn: str) -> dict:
    json_path = (REC_DIR / fn).with_suffix(".json")

    data = _read_json_data(json_path)
    if data is None:
        return {}  # or "MISSING/INVALID"

    # raw_flags_set = _extract_raw_flags_set(data)
    # print(json_path.stem, raw_flags_set)
    # if raw_flags_set is None:
    #     return ""  # invalid structure

    # return ",".join(sorted(raw_flags_set))  # stable order for printing

    stats = noises_annotated_stats(data)
    # print(stats)
    return stats

# for filename in file_names:
#     flags = flags_from_filename(filename)
#     print(f"{filename}: {flags}")

# from pathlib import Path
import pandas as pd

rows = []
for i, filename in enumerate(file_names, start=1):
    
    stats = noises_annotated_stats_from_filename(filename)
    rows.append({
        "nr": i,
        "filename": filename,
        "ann_nz_cnt": stats["noises_annotated_count"],
        "ann_nz_len": stats["noises_annotated_len_sum"]
    })

df = pd.DataFrame(rows, columns=["nr", "filename", "ann_nz_cnt", "ann_nz_len"])

# # If flags is a list/set/dict, convert to a readable string
# df["flags"] = df["flags"].apply(
#     lambda x: "|".join(map(str, x)) if isinstance(x, (list, tuple, set))
#     else (str(x) if x is not None else "")
# )

out_xlsx = Path("annotated_noises.xlsx")
df.to_excel(out_xlsx, index=False)

print(f"Wrote: {out_xlsx.resolve()}")

# matches = matches.copy()
# matches["flags"] = matches["filename"].astype(str).apply(flags_from_filename)


# cols_to_show = [c for c in ["filename", "recordingId", "userId", "tag", "basename", "flags"] if c in matches.columns]
# print("\n=== MATCHES (same basename) ===")
# print(matches[cols_to_show].to_string(index=False))

# Surandami unikalūs userId ir jų pasikartojimai
# user_counts_df = (
#     matches["userId"]
#     .dropna()
#     .astype(str)
#     .value_counts()
#     .rename_axis("userId")
#     .reset_index(name="count")
# )
# print(user_counts_df.head())
# print("\nSurandami unikalūs userId ir jų pasikartojimai:")
# print(user_counts_df.to_string(index=False))

# 6144c682bd0cc5acb727536 inmed20@zive.io


# print matching cases (choose the columns you want to see)
# cols_to_show = [c for c in ["filename", "recordingId", "tag", "basename"] if c in matches.columns]
# print("\n=== MATCHES (same basename) ===")
# print(matches[cols_to_show].to_string(index=False))

# If you also want to print CSV entries not found in Excel and Excel entries not in CSV, tell me and I’ll add the two “missing” lists (it’s ~10 extra lines).



['1626330.744', '1626924.927', '1626934.963', '1626941.468', '1630673.825', '1630693.635', '1630757.303', '1630757.924', '1630758.549', '1630759.17', '1630761.032', '1630762.273', '1630762.894', '1630763.516', '1630764.137', '1630768.486', '1630770.348', '1630771.589', '1630777.178', '1630844.567', '1630845.187', '1630846.425', '1630922.972', '1630923.598', '1630924.224', '1630924.849', '1630925.473', '1630926.102', '1630926.726', '1630927.98', '1630928.606', '1630930.48', '1630931.105', '1630931.73', '1630932.359', '1630932.984', '1630936.742', '1630940.501', '1630941.126', '1630941.751', '1630943.629', '1630944.253', '1630949.261', '1630951.771', '1630957.402', '1630958.03', '1630958.655', '1630959.28', '1630959.909', '1630960.534', '1630961.159', '1630961.788', '1630962.413', '1630963.038', '1630964.293', '1630964.918', '1630969.301', '1630969.93', '1630981.205', '1630981.83', '1630986.207', '1631003.727', '1631009.364', '1631009.724', '1631010.352', '1631011.609', '1631012.235', '1